# Evo-1 DiT Layer Pruning -- Experiment 2
> Reference: ExperimentalOverview.md

Phase 2 structured pruning from a trained checkpoint (cheaper than
Phase 1 retrain-from-scratch):

| Phase | Description | Runtime |
|---|---|---|
| 1 | Activation-change ratio analysis | ~15 min |
| 2 | Create pruned checkpoint | ~5 min |
| 3 | Evaluate pruned model (no FT) | ~4 hrs |
| 4 [OPTIONAL] | Stage-1 fine-tuning 5k steps | ~3 hrs |
| 5 | Post-FT evaluation | ~4 hrs |
| 6 | FLOP analysis + results summary | ~5 min |

## Why cross-attention-only DiT is different
In standard image DiTs, self-attention builds spatial structure across
layers -- depth matters a lot. In Evo-1's DiT each layer independently
refines action tokens using the fixed VLM KV-cache. No inter-token
self-attention. The marginal value of additional layers may fall off
fast. Hypothesis: 6-layer retains >95% of 8-layer performance.

## Hardware
A100 40 GB (Colab Pro/Pro+) or TC1 V100 32 GB.


In [ ]:
# ---- USER CONFIGURATION --------------------------------------------
WORKSPACE        = '/content/drive/MyDrive/evo1_study_pruning'
NUM_TRIALS       = 10   # parallel eval trials
N_RATIO_PASSES   = 50   # forward passes to average activation-change ratios
NUM_PRUNE_LAYERS = 2    # remove this many lowest-importance layers
USE_FLASH_ATTN   = True

LIBERO_BASE_PORTS    = list(range(9001, 9001 + NUM_TRIALS))
METAWORLD_BASE_PORTS = list(range(9011, 9011 + NUM_TRIALS))

# Path to evo1_complete.ipynb results (for 8-layer baseline comparison)
# Set to None to skip comparison.
BASELINE_RESULTS_PATH = '/content/drive/MyDrive/evo1_study/Results'
print('Config loaded.')


In [ ]:
from google.colab import drive
from pathlib import Path
import os, time, subprocess, json, re
import numpy as np

drive.mount('/content/drive')

RESULTS_DIR      = Path(f'{WORKSPACE}/Results')
ANALYSIS_DIR     = RESULTS_DIR / 'layer_analysis'
PRUNED_DIR       = RESULTS_DIR / 'pruned_eval'
FINETUNED_DIR    = RESULTS_DIR / 'finetuned_eval'
PLOTS_DIR        = RESULTS_DIR / 'plots'
CHECKPOINTS_DIR  = Path('/content/checkpoints')
LOGS_DIR         = Path('/content/logs')

for d in [
    RESULTS_DIR, ANALYSIS_DIR, PRUNED_DIR, PRUNED_DIR/'libero',
    PRUNED_DIR/'metaworld', FINETUNED_DIR, FINETUNED_DIR/'libero',
    FINETUNED_DIR/'metaworld', PLOTS_DIR, CHECKPOINTS_DIR, LOGS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

os.environ['MUJOCO_GL'] = 'egl'
os.environ.pop('DISPLAY', None)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU')
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name}  VRAM: {gpu.total_memory/1e9:.1f} GB')


In [ ]:
%%capture install_log
import subprocess

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
        raise RuntimeError('Failed: ' + cmd[:80])

# evo1_server (Python 3.10) ----------------------------------------
run('conda create -n evo1_server python=3.10 -y 2>/dev/null || true')
run('conda run -n evo1_server pip install -q '
    'torch==2.5.1 torchvision==0.20.1 '
    '--index-url https://download.pytorch.org/whl/cu121')
# CRITICAL: transformers must be 4.57.6, NOT 5.x
run('conda run -n evo1_server pip install -q transformers==4.57.6')
if USE_FLASH_ATTN:
    run('conda run -n evo1_server pip install -q flash-attn --no-build-isolation')
run('conda run -n evo1_server pip install -q '
    'accelerate diffusers einops timm websockets pyyaml h5py huggingface_hub')

# libero_client (Python 3.8.13 -- official LIBERO requirement) -----
run('conda create -n libero_client python=3.8.13 -y 2>/dev/null || true')
run('conda run -n libero_client pip install -q '
    'torch==1.11.0+cu113 torchvision==0.12.0+cu113 '
    '--extra-index-url https://download.pytorch.org/whl/cu113')
run('conda run -n libero_client pip install -q '
    'robosuite==1.4.1 mujoco==2.3.7 imageio h5py "bddl==0.2.7" websockets '
    'transformers==4.57.6')

# metaworld_client (Python 3.10) -----------------------------------
run('conda create -n metaworld_client python=3.10 -y 2>/dev/null || true')
run('conda run -n metaworld_client pip install -q '
    'mujoco metaworld gymnasium websockets opencv-python')

print('All three conda environments installed.')


In [ ]:
import subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

if not Path('/content/Evo-1').exists():
    subprocess.run('git clone --depth 1 https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1',
                   shell=True, check=True)
    subprocess.run('conda run -n evo1_server pip install -q -e /content/Evo-1',
                   shell=True, check=True)
    print('Evo-1 cloned')

if not Path('/content/LIBERO').exists():
    subprocess.run(
        'git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO',
        shell=True, check=True)
    subprocess.run('conda run -n libero_client pip install -q -e /content/LIBERO',
                   shell=True, check=True)
    print('LIBERO cloned')

for repo_id, local in [
    ('MINT-SJTU/Evo1_LIBERO',     CHECKPOINTS_DIR / 'libero'),
    ('MINT-SJTU/Evo1_MetaWorld',  CHECKPOINTS_DIR / 'metaworld'),
]:
    if not local.exists():
        print(f'Downloading {repo_id}...')
        snapshot_download(repo_id, local_dir=str(local), local_dir_use_symlinks=False)
        print(f'  done -> {local}')

print('Setup complete.')


In [ ]:
import subprocess, time, json, socket
from pathlib import Path
import numpy as np

def kill_ports(ports):
    for p in ports:
        subprocess.run(f'lsof -ti:{p} | xargs kill -9 2>/dev/null || true',
                       shell=True, capture_output=True)

def wait_for_port(port, timeout_sec=120, retry_sec=5):
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        try:
            s = socket.create_connection(('localhost', port), timeout=2)
            s.close(); return True
        except OSError:
            time.sleep(retry_sec)
    print(f'  Port {port} not ready after {timeout_sec}s')
    return False

def start_servers(cmd_tpl, ports, stagger=18, wait=180):
    procs = []
    for i, port in enumerate(ports, 1):
        cmd = cmd_tpl.format(port=port, idx=i)
        procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))
        if i < len(ports): time.sleep(stagger)
    print(f'  Waiting {wait}s for servers to init...')
    time.sleep(wait)
    return procs

def kill_servers(procs, ports):
    for p in procs:
        p.terminate()
        try: p.wait(timeout=10)
        except: p.kill()
    kill_ports(ports)

def wait_clients(procs, timeout_min=90, check_sec=30):
    deadline = time.time() + timeout_min * 60
    while any(p.poll() is None for p in procs):
        if time.time() > deadline:
            for p in procs:
                if p.poll() is None: p.kill()
            break
        n = sum(1 for p in procs if p.poll() is not None)
        print(f'  {n}/{len(procs)} done', end='\r')
        time.sleep(check_sec)
    print()

def collect_sr(result_dir, label, n):
    # Parse per-trial JSON output files into a list of success-rate floats.
    rates = []
    for i in range(1, n + 1):
        f = Path(result_dir) / f'{label}_trial{i}.json'
        if not f.exists(): continue
        try:
            data = json.loads(f.read_text())
            if isinstance(data, dict):
                sr = (data.get('success_rate') or data.get('mean_success')
                      or data.get('sr'))
                if sr is None and 'results' in data:
                    sr = sum(bool(r.get('success'))
                             for r in data['results']) / len(data['results'])
            elif isinstance(data, list):
                sr = sum(bool(r.get('success')) for r in data
                         if isinstance(r, dict)) / max(1, len(data))
            else:
                sr = float(data)
            rates.append(float(sr))
        except Exception as e:
            print(f'  Could not parse {f.name}: {e}')
    return rates

print('Helpers defined.')


---
## Phase 1: DiT Layer Importance Analysis

For each of the 8 DiT transformer blocks compute the activation-change ratio:
```
ratio_i = ||layer_output - layer_input||_2 / ||layer_input||_2
```
Low ratio -> layer changes input very little -> pruning candidate.
Averaged over N_RATIO_PASSES forward passes with synthetic observations.


In [ ]:
import sys, json, os, torch
import torch.nn as nn
sys.path.insert(0, '/content/Evo-1')
from Evo_1.models.EVO1 import EVO1

ckpt_dir = str(CHECKPOINTS_DIR / 'libero')
config   = json.load(open(os.path.join(ckpt_dir, 'config.json')))
stats    = json.load(open(os.path.join(ckpt_dir, 'norm_stats.json')))

config['finetune_vlm']            = False
config['finetune_action_head']    = False
config['num_inference_timesteps'] = 32

model = EVO1(config).eval().cuda()
ckpt_data = torch.load(
    os.path.join(ckpt_dir, 'mp_rank_00_model_states.pt'),
    map_location='cpu', weights_only=False,
)
model.load_state_dict(ckpt_data['module'], strict=True)
full_state = ckpt_data['module']  # keep for pruning later
n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {n_params/1e6:.0f} M params')

# Auto-discover the DiT transformer blocks (nn.ModuleList of length 8)
def find_dit_blocks(model):
    candidates = []
    for name, mod in model.named_modules():
        if not isinstance(mod, nn.ModuleList) or len(mod) != 8:
            continue
        first = list(mod.children())[0]
        if any(hasattr(first, a) for a in ['attn', 'cross_attn', 'norm1', 'norm2']):
            candidates.append((name, mod))
    if not candidates:
        # Broader fallback
        for name, mod in model.named_modules():
            if isinstance(mod, nn.ModuleList) and len(mod) == 8:
                candidates.append((name, mod))
    if not candidates:
        print('ModuleLists in model:')
        for n, m in model.named_modules():
            if isinstance(m, nn.ModuleList):
                print(f'  {n} len={len(m)}')
        raise RuntimeError(
            'Cannot auto-detect DiT blocks. '
            'Inspect the printout above and set dit_blocks manually.'
        )
    candidates.sort(key=lambda x: (0 if 'action_head' in x[0] else 1))
    return candidates[0]

block_path, dit_blocks = find_dit_blocks(model)
N_BLOCKS = len(dit_blocks)
block_name_in_path = block_path.split('.')[-1]
print(f'Found {N_BLOCKS} DiT blocks at: {block_path!r}')
print(f'First block type: {type(dit_blocks[0]).__name__}')


In [ ]:
# Register per-block pre/post hooks to capture input/output tensors
_hook_data = {i: {'inputs': [], 'outputs': []} for i in range(N_BLOCKS)}
_hooks = []

def _make_pre(idx):
    def _pre(module, args):
        x = args[0] if isinstance(args, tuple) else args
        _hook_data[idx]['inputs'].append(x.detach().float())
    return _pre

def _make_post(idx):
    def _post(module, args, output):
        x = output[0] if isinstance(output, tuple) else output
        _hook_data[idx]['outputs'].append(x.detach().float())
    return _post

for i, block in enumerate(dit_blocks):
    _hooks.append(block.register_forward_pre_hook(_make_pre(i)))
    _hooks.append(block.register_forward_hook(_make_post(i)))

print(f'Registered {len(_hooks)} hooks on {N_BLOCKS} DiT blocks.')


In [ ]:
from torch.amp import autocast

# Clear previous hook data
for i in range(N_BLOCKS):
    _hook_data[i]['inputs'].clear()
    _hook_data[i]['outputs'].clear()

image_mask  = torch.tensor([1, 1, 0], dtype=torch.int32, device='cuda')
action_mask = torch.tensor([1],       dtype=torch.int32, device='cuda')

print(f'Running {N_RATIO_PASSES} forward passes...')
with torch.no_grad():
    for p in range(N_RATIO_PASSES):
        imgs  = [torch.randn(3, 448, 448, device='cuda') for _ in range(3)]
        state = torch.zeros(1, 24, device='cuda')
        prompt = f'pick up the object and place it in the target {p}'
        with autocast(device_type='cuda', dtype=torch.bfloat16):
            _ = model.run_inference(
                images=imgs, image_mask=image_mask,
                prompt=prompt, state_input=state,
                action_mask=action_mask,
            )
        if (p + 1) % 10 == 0:
            print(f'  {p+1}/{N_RATIO_PASSES}', end='\r')

print(f'\nForward passes complete.')

# Compute activation-change ratios
layer_ratios = {}
for i in range(N_BLOCKS):
    ins  = _hook_data[i]['inputs']
    outs = _hook_data[i]['outputs']
    ratios = []
    for xi, xo in zip(ins, outs):
        if xi.shape == xo.shape:
            ratios.append(((xo - xi).norm() / (xi.norm() + 1e-8)).item())
        else:
            ratios.append((xo.norm() / (xi.norm() + 1e-8)).item())
    layer_ratios[i] = float(np.mean(ratios))

# Remove hooks
for h in _hooks: h.remove()

print('Layer activation-change ratios:')
for i, r in sorted(layer_ratios.items(), key=lambda x: x[1]):
    bar = '#' * int(r * 20)
    print(f'  Layer {i}: {r:.4f}  {bar}')

(ANALYSIS_DIR / 'layer_ratios.json').write_text(
    json.dumps({'layer_ratios': layer_ratios, 'n_passes': N_RATIO_PASSES}, indent=2)
)
print('Ratios saved.')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ratios = [layer_ratios[i] for i in range(N_BLOCKS)]

sorted_layers = sorted(range(N_BLOCKS), key=lambda i: layer_ratios[i])
prune_targets = sorted_layers[:NUM_PRUNE_LAYERS]
keep_layers   = [i for i in range(N_BLOCKS) if i not in prune_targets]

colors = ['#d62728' if i in prune_targets else '#1f77b4' for i in range(N_BLOCKS)]
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(N_BLOCKS), ratios, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('DiT Layer Index', fontsize=12)
ax.set_ylabel('Mean activation-change ratio\n||out-in|| / ||in||', fontsize=11)
ax.set_title('Evo-1 DiT Layer Importance  (Red = pruning targets)', fontsize=13)
ax.set_xticks(range(N_BLOCKS))
for i in prune_targets:
    ax.annotate('PRUNE', (i, ratios[i] + 0.002), ha='center',
                fontsize=9, color='#d62728', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'layer_importance.pdf', bbox_inches='tight')
fig.savefig(PLOTS_DIR / 'layer_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pruning plan:')
print(f'  Remove layers: {prune_targets}  '
      f'(ratios: {[round(layer_ratios[i],4) for i in prune_targets]})')
print(f'  Keep   layers: {keep_layers}')
print(f'  Result: {N_BLOCKS - NUM_PRUNE_LAYERS}-layer DiT')


---
## Phase 2: Create Pruned Checkpoint

Remove the lowest-importance blocks and remap the remaining block keys
to sequential indices 0 ... (N-1).
This produces a valid `(8 - NUM_PRUNE_LAYERS)`-layer checkpoint.


In [ ]:
import torch, re, json, os

def remap_pruned_state(state_dict, keep_idx, blk_name):
    # Reindex kept block keys: old index -> new sequential index
    old_to_new = {old: new for new, old in enumerate(sorted(keep_idx))}
    new_state = {}
    pattern = re.compile(rf'(.+{blk_name}\.)([0-9]+)(\..*)' )
    for key, val in state_dict.items():
        m = pattern.match(key)
        if m:
            oi = int(m.group(2))
            if oi in old_to_new:
                new_key = f'{m.group(1)}{old_to_new[oi]}{m.group(3)}'
                new_state[new_key] = val
            # else: skip pruned layer
        else:
            new_state[key] = val
    return new_state

pruned_state = remap_pruned_state(full_state, keep_layers, block_name_in_path)

n_orig   = len([k for k in full_state   if block_name_in_path + '.' in k])
n_pruned = len([k for k in pruned_state if block_name_in_path + '.' in k])
print(f'State-dict block keys: {n_orig} -> {n_pruned}')

pruned_config = dict(config)
pruned_config['num_layers'] = N_BLOCKS - NUM_PRUNE_LAYERS
print(f'Pruned model num_layers = {pruned_config["num_layers"]}')
print(f'Keep indices: {keep_layers}  |  Prune indices: {prune_targets}')


In [ ]:
import shutil
from pathlib import Path

pruned_ckpt_dir = CHECKPOINTS_DIR / 'libero_pruned'
pruned_ckpt_dir.mkdir(parents=True, exist_ok=True)

torch.save({'module': pruned_state}, str(pruned_ckpt_dir / 'mp_rank_00_model_states.pt'))
(pruned_ckpt_dir / 'config.json').write_text(json.dumps(pruned_config, indent=2))
shutil.copy(CHECKPOINTS_DIR / 'libero' / 'norm_stats.json',
            pruned_ckpt_dir / 'norm_stats.json')

# Save to Drive for persistence
drive_ckpt = Path(f'{WORKSPACE}/checkpoints/libero_pruned')
drive_ckpt.mkdir(parents=True, exist_ok=True)
for f in pruned_ckpt_dir.iterdir():
    shutil.copy(f, drive_ckpt / f.name)

# Save pruning metadata
meta = {
    'prune_targets': prune_targets,
    'keep_layers': keep_layers,
    'n_layers_original': N_BLOCKS,
    'n_layers_pruned': N_BLOCKS - NUM_PRUNE_LAYERS,
    'layer_ratios': layer_ratios,
}
(ANALYSIS_DIR / 'pruning_metadata.json').write_text(json.dumps(meta, indent=2))
print(f'Pruned checkpoint saved: {pruned_ckpt_dir}')
print(f'Drive copy: {drive_ckpt}')

# Free GPU memory before evaluation servers start
del model
torch.cuda.empty_cache()
print('GPU memory freed.')


---
## Phase 3: Quick Evaluation of Pruned Model (No Fine-tuning)

Run 10-trial evaluation of the pruned model on LIBERO and MetaWorld.
This shows the baseline degradation without fine-tuning and informs
whether Phase 4 fine-tuning is worth pursuing.


In [ ]:
%%writefile /content/evo1_pruned_server.py
#!/usr/bin/env python3
# Evo-1 inference server for a pruned (fewer-layer) DiT model.
# Loads the remapped checkpoint produced by the pruning notebook.
# Usage:
#   python evo1_pruned_server.py --checkpoint /content/checkpoints/libero_pruned --port 9001
import asyncio, json, os, sys, argparse
import numpy as np
import torch

sys.path.insert(0, '/content/Evo-1')
from Evo_1.models.EVO1 import EVO1

parser = argparse.ArgumentParser()
parser.add_argument("--checkpoint", type=str, required=True,
                    help="Path to pruned checkpoint directory (contains config.json with num_layers)")
parser.add_argument("--port",  type=int, required=True)
parser.add_argument("--name",  type=str, default="pruned_server")
args = parser.parse_args()

class Normalizer:
    def __init__(self, stats):
        self.state_min  = torch.tensor(stats["state_min"],  dtype=torch.float32)
        self.state_max  = torch.tensor(stats["state_max"],  dtype=torch.float32)
        self.action_min = torch.tensor(stats["action_min"], dtype=torch.float32)
        self.action_max = torch.tensor(stats["action_max"], dtype=torch.float32)
    def normalize_state(self, s):
        if s.ndim == 1: s = s.view(1, -1)
        smin = self.state_min.to(s.device, dtype=s.dtype)
        smax = self.state_max.to(s.device, dtype=s.dtype)
        return (s - smin) / (smax - smin + 1e-8) * 2.0 - 1.0
    def denormalize_action(self, a):
        if a.ndim == 1: a = a.view(1, -1)
        amin = self.action_min.to(a.device, dtype=a.dtype)
        amax = self.action_max.to(a.device, dtype=a.dtype)
        return (a + 1.0) / 2.0 * (amax - amin + 1e-8) + amin

def load_pruned_model(ckpt_dir):
    # config.json in the pruned dir has num_layers = (8 - NUM_PRUNE_LAYERS)
    config = json.load(open(os.path.join(ckpt_dir, "config.json")))
    stats  = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))
    config["finetune_vlm"]            = False
    config["finetune_action_head"]    = False
    config["num_inference_timesteps"] = 32
    n_layers = config.get("num_layers", "?")
    print(f"[{args.name}] Loading {n_layers}-layer DiT from {ckpt_dir}")
    model = EVO1(config).eval()
    ckpt  = torch.load(os.path.join(ckpt_dir, "mp_rank_00_model_states.pt"),
                       map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["module"], strict=True)
    return model.to("cuda"), Normalizer(stats)

model, normalizer = load_pruned_model(args.checkpoint)

def decode_image(img_list):
    arr = np.array(img_list, dtype=np.uint8)
    if arr.ndim == 3 and arr.shape[2] == 3:
        arr = arr.transpose(2, 0, 1)
    return torch.from_numpy(arr).float().to("cuda") / 255.0

def infer(data):
    images      = [decode_image(img) for img in data["image"]]
    state       = torch.tensor(data["state"], dtype=torch.float32, device="cuda")
    if state.ndim == 1: state = state.unsqueeze(0)
    if state.shape[1] < 24:
        state = torch.cat([state, torch.zeros(1, 24 - state.shape[1], device="cuda")], 1)
    norm_state  = normalizer.normalize_state(state).to(torch.float32)
    image_mask  = torch.tensor(data["image_mask"],   dtype=torch.int32, device="cuda")
    action_mask = torch.tensor([data["action_mask"]], dtype=torch.int32, device="cuda")
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        action = model.run_inference(
            images=images, image_mask=image_mask,
            prompt=data["prompt"], state_input=norm_state,
            action_mask=action_mask,
        )
    action = action.reshape(1, -1, 24)
    return normalizer.denormalize_action(action[0]).cpu().numpy().tolist()

import websockets

async def handle(ws):
    try:
        async for msg in ws:
            await ws.send(json.dumps(infer(json.loads(msg))))
    except websockets.exceptions.ConnectionClosed:
        pass
    except Exception:
        import traceback; traceback.print_exc()

async def main():
    print(f"[{args.name}] port={args.port}")
    async with websockets.serve(
        handle, "0.0.0.0", args.port,
        max_size=100_000_000, ping_interval=120, ping_timeout=300,
    ):
        await asyncio.Future()

asyncio.run(main())


In [ ]:
import shutil, torch, re, json
from pathlib import Path

# Build pruned MetaWorld checkpoint using the same pruning plan
mw_full = torch.load(
    str(CHECKPOINTS_DIR / 'metaworld' / 'mp_rank_00_model_states.pt'),
    map_location='cpu', weights_only=False
)['module']

mw_pruned = remap_pruned_state(mw_full, keep_layers, block_name_in_path)

pruned_mw_dir = CHECKPOINTS_DIR / 'metaworld_pruned'
pruned_mw_dir.mkdir(parents=True, exist_ok=True)
torch.save({'module': mw_pruned},
           str(pruned_mw_dir / 'mp_rank_00_model_states.pt'))

mw_cfg = json.loads((CHECKPOINTS_DIR / 'metaworld' / 'config.json').read_text())
mw_cfg['num_layers'] = N_BLOCKS - NUM_PRUNE_LAYERS
(pruned_mw_dir / 'config.json').write_text(json.dumps(mw_cfg, indent=2))
shutil.copy(CHECKPOINTS_DIR / 'metaworld' / 'norm_stats.json',
            pruned_mw_dir / 'norm_stats.json')
print('MetaWorld pruned checkpoint ready.')


In [ ]:
# LIBERO eval of pruned model
print('Starting pruned LIBERO servers (10 x ports 9001-9010)...')
kill_ports(LIBERO_BASE_PORTS)
time.sleep(2)

srv_cmd = (
    'conda run -n evo1_server python /content/evo1_pruned_server.py '
    f'--checkpoint {CHECKPOINTS_DIR}/libero_pruned '
    '--port {port} --name pruned_libero_{idx} '
    f'> {LOGS_DIR}/pruned_libero_{{port}}.log 2>&1'
)
srv_procs = start_servers(srv_cmd, LIBERO_BASE_PORTS)

client_procs = []
for i, port in enumerate(LIBERO_BASE_PORTS, 1):
    out = PRUNED_DIR / 'libero' / f'pruned_trial{i}.json'
    cmd = (
        'export PYTHONPATH=/content/LIBERO && '
        'conda run -n libero_client python '
        '/content/LIBERO/scripts/libero_eval.py '
        '--server-address localhost '
        f'--server-port {port} '
        '--benchmark libero_90 --num-episodes 50 '
        f'--output {out} '
        f'> {LOGS_DIR}/pruned_libero_client_{port}.log 2>&1'
    )
    client_procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))

wait_clients(client_procs, timeout_min=90)
kill_servers(srv_procs, LIBERO_BASE_PORTS)

pruned_libero_rates = collect_sr(PRUNED_DIR / 'libero', 'pruned', NUM_TRIALS)
m, s = np.mean(pruned_libero_rates), np.std(pruned_libero_rates)
print(f'Pruned LIBERO SR: {m:.3f} +/- {s:.3f}  (n={len(pruned_libero_rates)})')
(PRUNED_DIR / 'libero_summary.json').write_text(
    json.dumps({'rates': pruned_libero_rates,
                'mean': float(m) if pruned_libero_rates else None,
                'std':  float(s) if pruned_libero_rates else None}, indent=2)
)


In [ ]:
# MetaWorld eval of pruned model
print('Starting pruned MetaWorld servers (10 x ports 9011-9020)...')
kill_ports(METAWORLD_BASE_PORTS)
time.sleep(2)

srv_cmd = (
    'conda run -n evo1_server python /content/evo1_pruned_server.py '
    f'--checkpoint {CHECKPOINTS_DIR}/metaworld_pruned '
    '--port {port} --name pruned_mw_{idx} '
    f'> {LOGS_DIR}/pruned_mw_{{port}}.log 2>&1'
)
srv_procs = start_servers(srv_cmd, METAWORLD_BASE_PORTS)

client_procs = []
for i, port in enumerate(METAWORLD_BASE_PORTS, 1):
    out = PRUNED_DIR / 'metaworld' / f'pruned_trial{i}.json'
    cmd = (
        'conda run -n metaworld_client python '
        '/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_prompt.py '
        f'--server-port {port} '
        f'--output {out} '
        f'> {LOGS_DIR}/pruned_mw_client_{port}.log 2>&1'
    )
    client_procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))

wait_clients(client_procs, timeout_min=60)
kill_servers(srv_procs, METAWORLD_BASE_PORTS)

pruned_mw_rates = collect_sr(PRUNED_DIR / 'metaworld', 'pruned', NUM_TRIALS)
m, s = np.mean(pruned_mw_rates), np.std(pruned_mw_rates)
print(f'Pruned MetaWorld SR: {m:.3f} +/- {s:.3f}  (n={len(pruned_mw_rates)})')
(PRUNED_DIR / 'metaworld_summary.json').write_text(
    json.dumps({'rates': pruned_mw_rates,
                'mean': float(m) if pruned_mw_rates else None,
                'std':  float(s) if pruned_mw_rates else None}, indent=2)
)


---
## Phase 4: Stage-1 Fine-tuning  [OPTIONAL -- TC1 RECOMMENDED]

> **Estimated runtime:** ~3 hours (5000 training steps on A100) + data download.
> Colab free tier will hit the session limit -- use Colab Pro+ or TC1.

Fine-tune the pruned model for 5000 steps (Stage-1 equivalent).
Only the `action_head` is updated; the VLM backbone is frozen.

Steps:
1. Download LIBERO training data
2. Check `train.py --help` to confirm argument names
3. Run Stage-1 fine-tuning with `accelerate launch`
4. Copy checkpoint to Drive

If fine-tuning is skipped, Phase 3 results (pruned, no FT) vs the
8-layer baseline from `evo1_complete.ipynb` still show the degradation
upper bound and inform whether fine-tuning is necessary.


In [ ]:
# [OPTIONAL] Download LIBERO training data
from pathlib import Path
from huggingface_hub import snapshot_download

TRAIN_DATA_DIR = Path('/content/data/libero_train')

if not TRAIN_DATA_DIR.exists():
    print('Downloading LIBERO training data (may take 20-60 min)...')
    TRAIN_DATA_DIR.mkdir(parents=True, exist_ok=True)
    try:
        snapshot_download('MINT-SJTU/Evo1_LIBERO_data',
                          local_dir=str(TRAIN_DATA_DIR),
                          local_dir_use_symlinks=False)
        print('LIBERO training data downloaded (MINT-SJTU source)')
    except Exception as e:
        print(f'  Primary source failed: {e}')
        print('  Trying LIBERO official dataset...')
        snapshot_download('Lifelong-Robot-Learning/LIBERO_data',
                          local_dir=str(TRAIN_DATA_DIR),
                          local_dir_use_symlinks=False)
        print('LIBERO data downloaded (official source)')
else:
    print(f'Training data present: {TRAIN_DATA_DIR}')

# Print train.py help BEFORE running so args can be verified
import subprocess
r = subprocess.run(
    'conda run -n evo1_server python /content/Evo-1/scripts/train.py --help',
    shell=True, capture_output=True, text=True)
print('=== train.py --help ===')
print(r.stdout[:3000])
if r.stderr: print('STDERR:', r.stderr[:500])


In [ ]:
# [OPTIONAL] Run Stage-1 fine-tuning (5000 steps, ~3 hrs)
# IMPORTANT: Verify argument names from --help output above before running.
import subprocess, time, shutil
from pathlib import Path

ft_output_dir = Path('/content/checkpoints/libero_pruned_ft')
ft_output_dir.mkdir(parents=True, exist_ok=True)

# Adjust args based on train.py --help output
train_cmd = (
    'conda run -n evo1_server accelerate launch '
    '/content/Evo-1/scripts/train.py '
    f'--num_layers {N_BLOCKS - NUM_PRUNE_LAYERS} '
    f'--checkpoint {CHECKPOINTS_DIR}/libero_pruned '
    f'--data_path {TRAIN_DATA_DIR} '
    '--stage 1 '
    '--max_steps 5000 '
    f'--output_dir {ft_output_dir} '
    f'> {LOGS_DIR}/finetune_stage1.log 2>&1'
)
print('Fine-tuning command:')
print(' ', train_cmd[:300])
print('\nStarting fine-tuning (~3 hrs)...')

proc = subprocess.Popen(train_cmd, shell=True, executable='/bin/bash')
start = time.time()
while proc.poll() is None:
    print(f'  Training... {(time.time()-start)/60:.1f} min', end='\r')
    time.sleep(60)

rc = proc.returncode
if rc == 0:
    print(f'\nFine-tuning done in {(time.time()-start)/60:.0f} min')
    # Save to Drive
    drive_ft = Path(f'{WORKSPACE}/checkpoints/libero_pruned_ft')
    drive_ft.mkdir(parents=True, exist_ok=True)
    for f in ft_output_dir.iterdir():
        shutil.copy(f, drive_ft / f.name)
    print(f'Checkpoint -> Drive: {drive_ft}')
else:
    print(f'\nFine-tuning failed (rc={rc}). Check {LOGS_DIR}/finetune_stage1.log')


---
## Phase 5: Evaluation of Fine-tuned Model  [OPTIONAL]


In [ ]:
# [OPTIONAL] Evaluate fine-tuned pruned model
import shutil, subprocess
from pathlib import Path

ft_ckpt = Path('/content/checkpoints/libero_pruned_ft')
if not ft_ckpt.exists():
    drive_ft = Path(f'{WORKSPACE}/checkpoints/libero_pruned_ft')
    if drive_ft.exists():
        ft_ckpt.mkdir(parents=True, exist_ok=True)
        for f in drive_ft.iterdir():
            shutil.copy(f, ft_ckpt / f.name)
    else:
        raise FileNotFoundError('Fine-tuned checkpoint not found. Run Phase 4 first.')

# LIBERO
print('Starting fine-tuned LIBERO servers...')
kill_ports(LIBERO_BASE_PORTS)
time.sleep(2)
srv_procs = start_servers(
    'conda run -n evo1_server python /content/evo1_pruned_server.py '
    f'--checkpoint {ft_ckpt} '
    '--port {port} --name ft_libero_{idx} '
    f'> {LOGS_DIR}/ft_libero_{{port}}.log 2>&1',
    LIBERO_BASE_PORTS,
)
client_procs = []
for i, port in enumerate(LIBERO_BASE_PORTS, 1):
    out = FINETUNED_DIR / 'libero' / f'ft_trial{i}.json'
    cmd = (
        'export PYTHONPATH=/content/LIBERO && '
        'conda run -n libero_client python '
        '/content/LIBERO/scripts/libero_eval.py '
        '--server-address localhost '
        f'--server-port {port} '
        '--benchmark libero_90 --num-episodes 50 '
        f'--output {out} '
        f'> {LOGS_DIR}/ft_libero_client_{port}.log 2>&1'
    )
    client_procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))
wait_clients(client_procs, timeout_min=90)
kill_servers(srv_procs, LIBERO_BASE_PORTS)

ft_libero_rates = collect_sr(FINETUNED_DIR / 'libero', 'ft', NUM_TRIALS)
m, s = np.mean(ft_libero_rates), np.std(ft_libero_rates)
print(f'Fine-tuned LIBERO SR: {m:.3f} +/- {s:.3f}')
(FINETUNED_DIR / 'libero_summary.json').write_text(
    json.dumps({'rates': ft_libero_rates,
                'mean': float(m) if ft_libero_rates else None,
                'std':  float(s) if ft_libero_rates else None}, indent=2)
)


---
## FLOP Analysis

```
DiT FLOPs (per action chunk) ~ NFE x num_layers x 2 x seq_len x hidden_dim^2
where seq_len = horizon = 50
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Estimate hidden_dim from state dict
q_keys = [k for k in full_state if 'q_proj.weight' in k
          or 'to_q.weight' in k or 'query.weight' in k]
hidden_dim = full_state[q_keys[0]].shape[0] if q_keys else 256
print(f'Estimated hidden_dim: {hidden_dim}')

def dit_flops(nfe, n_layers, seq=50, h=hidden_dim):
    return nfe * n_layers * 2 * seq * h ** 2

NFE_DEFAULT = 32
n_pruned = N_BLOCKS - NUM_PRUNE_LAYERS

configs = [
    (f'8L baseline (NFE={NFE_DEFAULT})', NFE_DEFAULT, 8),
    (f'8L  (NFE=10)',                    10,          8),
    (f'8L  (NFE=2)',                      2,          8),
    (f'{n_pruned}L pruned (NFE={NFE_DEFAULT})',  NFE_DEFAULT, n_pruned),
    (f'{n_pruned}L pruned (NFE=2)',               2,          n_pruned),
    ('4L  (NFE=2)',                       2,          4),
    ('2L  (NFE=1)',                       1,          2),
]

base_f = dit_flops(NFE_DEFAULT, 8)
print(f'\n{"Config":<38} {"FLOPs (rel)":<14} {"Speedup"}')
print('-' * 60)
for lbl, nfe, nl in configs:
    f = dit_flops(nfe, nl)
    print(f'  {lbl:<36} {f/base_f:.3f}x         {base_f/f:.1f}x')

labels   = [c[0] for c in configs]
speedups = [base_f / dit_flops(c[1], c[2]) for c in configs]
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(labels[::-1], speedups[::-1], color='steelblue')
ax.set_xlabel('DiT compute speedup (x 8L NFE=32 baseline)', fontsize=11)
ax.set_title('Estimated DiT FLOP Reduction by Configuration', fontsize=12)
ax.axvline(1, linestyle='--', color='gray')
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'flop_comparison.pdf', bbox_inches='tight')
fig.savefig(PLOTS_DIR / 'flop_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('flop_comparison saved')


---
## Results Summary


In [ ]:
import json, numpy as np
from pathlib import Path

def _load_sr(path):
    p = Path(path)
    if not p.exists(): return None, None
    d = json.loads(p.read_text())
    return d.get('mean'), d.get('std')

bl_l_m, bl_l_s = _load_sr(f'{BASELINE_RESULTS_PATH}/baseline/libero_baseline_summary.json')
bl_m_m, bl_m_s = _load_sr(f'{BASELINE_RESULTS_PATH}/baseline/metaworld_baseline_summary.json')
pr_l_m, pr_l_s = _load_sr(PRUNED_DIR / 'libero_summary.json')
pr_m_m, pr_m_s = _load_sr(PRUNED_DIR / 'metaworld_summary.json')
ft_l_m, ft_l_s = _load_sr(FINETUNED_DIR / 'libero_summary.json')

n_pruned = N_BLOCKS - NUM_PRUNE_LAYERS
sp = f'{N_BLOCKS/n_pruned:.2f}x'

def fmt(m, s):
    return f'{m:.1%} +/- {s:.1%}' if m is not None else '---'

print('=' * 72)
print(f'{"Model":<35} {"LIBERO SR":>17} {"MetaWorld SR":>14} {"FLOPs":>6}')
print('-' * 72)
print(f'  {"8-layer baseline":<33} {fmt(bl_l_m, bl_l_s):>17} '
      f'{fmt(bl_m_m, bl_m_s):>14} {"1.00x":>6}')
print(f'  {f"{n_pruned}L pruned (no FT)":<33} {fmt(pr_l_m, pr_l_s):>17} '
      f'{fmt(pr_m_m, pr_m_s):>14} {sp:>6}')
if ft_l_m is not None:
    print(f'  {f"{n_pruned}L pruned + FT":<33} {fmt(ft_l_m, ft_l_s):>17} '
          f'{"---":>14} {sp:>6}')
print('=' * 72)

if pr_l_m and bl_l_m:
    ratio = pr_l_m / bl_l_m
    verdict = 'CONFIRMED (>95%)' if ratio > 0.95 else 'NOT confirmed (<95%)'
    print(f'\nHypothesis ({n_pruned}L retains >95% of 8L): {ratio:.1%} -> {verdict}')

summary = {
    'pruning': {'prune_targets': prune_targets, 'keep_layers': keep_layers,
                'n_layers_orig': N_BLOCKS, 'n_layers_pruned': n_pruned},
    'layer_ratios': layer_ratios,
    'results': {
        '8l_baseline_libero': {'mean': bl_l_m, 'std': bl_l_s},
        '8l_baseline_mw':     {'mean': bl_m_m, 'std': bl_m_s},
        'pruned_noft_libero': {'mean': pr_l_m, 'std': pr_l_s},
        'pruned_noft_mw':     {'mean': pr_m_m, 'std': pr_m_s},
        'pruned_ft_libero':   {'mean': ft_l_m, 'std': ft_l_s},
    },
}
(RESULTS_DIR / 'pruning_summary.json').write_text(json.dumps(summary, indent=2))
print(f'\nResults saved to {RESULTS_DIR}')
